# c_prepare

Run all cells. Outputs are written to this task's `output/` folder.


In [1]:
%run ~/Desktop/SHL_dblp_comparable/common/core.ipynb


/Users/slmagid/miniforge3/envs/research313/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.home() / 'Desktop' / 'SHL_dblp_comparable'
CONFIG = read_config(ROOT)
UP = ROOT / 'b_validate' / 'output'
OUTPUT = ROOT / 'c_prepare' / 'output'
OUTPUT.mkdir(parents=True, exist_ok=True)
panel = pd.read_csv(UP / 'validated_panel.csv.gz')
people = pd.read_csv(UP / 'person_cohorts.csv')
cohorts = ['all_eligible_cropped13', 'complete13_cropped', 'legacy_complete20_cropped13']
aarc_root = Path.home() / 'Desktop' / 'SHL_aarc_modeling'
crosswalk = aarc_root / 'b_validate' / 'output' / 'person_id_crosswalk.csv.gz'
overlap = set()
if crosswalk.exists():
    overlap = set(pd.read_csv(crosswalk, usecols=['source_person_id']).source_person_id.astype(str))
    people['all_eligible_no_aarc_overlap'] = ~people.source_person_id.astype(str).isin(overlap)
    if int(people['all_eligible_no_aarc_overlap'].sum()) >= int(CONFIG['minimum_sensitivity_cohort_people']):
        cohorts.append('all_eligible_no_aarc_overlap')
parts = []
length_parts = []
for cohort in cohorts:
    ids = set(people.loc[people[cohort], 'source_person_id'].astype(str))
    p = panel[panel.source_person_id.astype(str).isin(ids)].copy()
    p['source_person_id'] = p.source_person_id.astype(str)
    p['person_domain_id'] = p.source_person_id + '::DBLP-CS'
    nxt = p[['person_domain_id', 'age', 'q']].copy()
    nxt['age'] -= 1
    nxt = nxt.rename(columns={'q': 'q_next'})
    t = p.merge(nxt, on=['person_domain_id', 'age'], how='left').dropna(subset=['q_next']).copy()
    t['analysis_group'] = cohort
    t['domain'] = 'Computer Science (DBLP)'
    t['transition_age'] = t.age.astype(int)
    t['target_age'] = t.transition_age + 1
    t['q_prev'] = t.q.astype(float)
    t['q'] = t.q_next.astype(float)
    t['stage'] = t.transition_age.map(stage_for_transition)
    t['active'] = (t.q > 0).astype(int)
    t['prev_active'] = (t.q_prev > 0).astype(int)
    t['restart'] = (t.q_prev <= 0).astype(int)
    t = add_raw_lags(t[['analysis_group', 'domain', 'source_person_id', 'person_domain_id', 'split', 'transition_age', 'target_age', 'stage', 'q_prev', 'q', 'active', 'prev_active', 'restart']], AR_ORDER)
    parts.append(t)
    lens = t.groupby(['analysis_group', 'person_domain_id', 'source_person_id', 'split'], as_index=False).target_age.max().rename(columns={'target_age': 'observed_years'})
    lens['observed_years'] += 1
    length_parts.append(lens)
rows = pd.concat(parts, ignore_index=True)
lengths = pd.concat(length_parts, ignore_index=True)
rows.to_csv(OUTPUT / 'modeling_transitions.csv.gz', index=False, compression='gzip')
lengths.to_csv(OUTPUT / 'trajectory_lengths.csv', index=False)
support = []
for (group, stage), data in rows.groupby(['analysis_group', 'stage']):
    rec = {'analysis_group': group, 'stage': stage, 'transitions': len(data), 'people': data.person_domain_id.nunique(), 'active_outcomes': int(data.active.sum())}
    for lag in range(1, AR_ORDER + 1):
        d = data[data[f'lag{lag}_available']]
        rec[f'lag{lag}_people'] = d.person_domain_id.nunique()
        rec[f'lag{lag}_transitions'] = len(d)
    support.append(rec)
pd.DataFrame(support).to_csv(OUTPUT / 'lag_support.csv', index=False)
overlap_report = {'aarc_crosswalk_found': crosswalk.exists(), 'aarc_unique_people': len(overlap), 'dblp_unique_people': int(people.source_person_id.nunique()), 'overlapping_source_ids': int(people.source_person_id.astype(str).isin(overlap).sum()), 'no_overlap_cohort_created': 'all_eligible_no_aarc_overlap' in cohorts, 'minimum_sensitivity_cohort_people': int(CONFIG['minimum_sensitivity_cohort_people'])}
write_json(OUTPUT / 'overlap_report.json', overlap_report)
write_json(OUTPUT / 'quality_report.json', {'status': 'passed', 'cohorts': {c: int(people[c].sum()) for c in cohorts}, 'raw_lags': list(range(1, 7)), 'memory_lags': [2, 3, 4, 5, 6], 'overlap': overlap_report})
print(lengths.groupby('analysis_group').observed_years.agg(['count', 'mean', 'min', 'max']).to_string())
